# Flower Classification - Model Training
## Training Multiple Classification Models

This notebook trains and compares multiple ML models for flower classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Load Preprocessed Data

In [ ]:
# Load preprocessed data
data_dir = Path('../data/processed')

X_train = np.load(data_dir / 'X_train.npy')
X_test = np.load(data_dir / 'X_test.npy')
y_train = np.load(data_dir / 'y_train.npy')
y_test = np.load(data_dir / 'y_test.npy')

# Load label encoder for class names
label_encoder = joblib.load(Path('../models/artifacts/label_encoder.joblib'))
class_names = label_encoder.classes_

print(f'Training data shape: {X_train.shape}')
print(f'Test data shape: {X_test.shape}')
print(f'Classes: {class_names}')

## 2. Initialize Models

In [ ]:
# Initialize all models
models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000,
        multi_class='multinomial'
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        random_state=42,
        n_estimators=100
    ),
    'XGBoost': xgb.XGBClassifier(
        random_state=42,
        n_estimators=100,
        eval_metric='mlogloss',
        use_label_encoder=False
    )
}

print(f'Initialized {len(models)} models:')
for name in models:
    print(f'  - {name}')

## 3. Train All Models

In [ ]:
# Train all models and store results
trained_models = {}
train_scores = {}
test_scores = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    train_scores[name] = model.score(X_train, y_train)
    test_scores[name] = model.score(X_test, y_test)
    print(f'  Train accuracy: {train_scores[name]:.4f}')
    print(f'  Test accuracy: {test_scores[name]:.4f}')
    print()

## 4. Cross-Validation

In [ ]:
# Perform 5-fold cross-validation
cv_scores = {}

for name, model in trained_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores[name] = {
        'mean': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }
    print(f'{name}: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})')

# Store CV results
cv_results_df = pd.DataFrame({
    'Model': list(cv_scores.keys()),
    'CV Mean': [cv_scores[k]['mean'] for k in cv_scores],
    'CV Std': [cv_scores[k]['std'] for k in cv_scores]
})
cv_results_df

## 5. Hyperparameter Tuning - Best Model

In [ ]:
# Identify best model based on test accuracy
best_model_name = max(test_scores, key=test_scores.get)
print(f'Best model so far: {best_model_name} (Test Accuracy: {test_scores[best_model_name]:.4f})')

# Define parameter grids for tuning
param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['lbfgs', 'saga'],
        'penalty': ['l2']
    },
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 20, None],
        'min_samples_split': [2, 5]
    },
    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2]
    }
}

In [ ]:
# Tune the best model
if best_model_name in param_grids:
    print(f'Tuning hyperparameters for {best_model_name}...')
    
    grid_search = GridSearchCV(
        trained_models[best_model_name],
        param_grids[best_model_name],
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f'\nBest parameters: {grid_search.best_params_}')
    print(f'Best CV score: {grid_search.best_score_:.4f}')
    
    # Update the model with best parameters
    trained_models[best_model_name] = grid_search.best_estimator_
    
    # Update test score
    test_scores[best_model_name] = grid_search.best_estimator_.score(X_test, y_test)
    print(f'Updated test score: {test_scores[best_model_name]:.4f}')
else:
    print(f'No parameter grid defined for {best_model_name}')

## 6. Results Comparison

In [ ]:
# Create comparison dataframe
results_df = pd.DataFrame({
    'Model': list(test_scores.keys()),
    'Train Accuracy': [train_scores[k] for k in test_scores],
    'Test Accuracy': [test_scores[k] for k in test_scores],
    'CV Mean': [cv_scores[k]['mean'] for k in test_scores],
    'CV Std': [cv_scores[k]['std'] for k in test_scores]
})

results_df = results_df.sort_values('Test Accuracy', ascending=False)
results_df

In [ ]:
# Plot accuracy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax1 = axes[0]
models_idx = range(len(results_df))
bars = ax1.bar(models_idx, results_df['Test Accuracy'], color=sns.color_palette('viridis', len(results_df)))
ax1.set_xticks(models_idx)
ax1.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax1.set_ylabel('Accuracy')
ax1.set_title('Test Accuracy Comparison')
ax1.set_ylim(0, 1.05)

for bar, val in zip(bars, results_df['Test Accuracy']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

# Line plot with train vs test
ax2 = axes[1]
x = np.arange(len(results_df))
width = 0.35

ax2.bar(x - width/2, results_df['Train Accuracy'], width, label='Train', color='steelblue')
ax2.bar(x + width/2, results_df['Test Accuracy'], width, label='Test', color='coral')
ax2.set_xticks(x)
ax2.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax2.set_ylabel('Accuracy')
ax2.set_title('Train vs Test Accuracy')
ax2.legend()
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 7. Feature Importance (Tree-based Models)

In [ ]:
# Load feature columns
feature_columns = joblib.load(Path('../models/artifacts/feature_columns.joblib'))

# Plot feature importance for Random Forest
if 'Random Forest' in trained_models:
    rf_model = trained_models['Random Forest']
    importances = rf_model.feature_importances_
    
    # Create dataframe
    importance_df = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
    plt.title('Random Forest - Feature Importance')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    print('Top 5 most important features:')
    for idx, row in importance_df.head(5).iterrows():
        print(f'  {row["Feature"]}: {row["Importance"]:.4f}')

## 8. Save Best Model

In [ ]:
# Save all trained models
models_dir = Path('../models/artifacts')
models_dir.mkdir(parents=True, exist_ok=True)

for name, model in trained_models.items():
    model_path = models_dir / f'{name.lower().replace(" ", "_")}_model.joblib'
    joblib.dump(model, model_path)
    print(f'Saved {name} to {model_path.name}')

# Save results
results_df.to_csv(models_dir / 'model_comparison_results.csv', index=False)
print(f'\nSaved comparison results to model_comparison_results.csv')

# Identify and save best model separately
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
joblib.dump(best_model, models_dir / 'best_model.joblib')
print(f'Best model: {best_model_name}')
print(f'Saved best model to best_model.joblib')

## Summary

✅ Trained **4 classification models**: Logistic Regression, Decision Tree, Random Forest, XGBoost  
✅ Performed **5-fold cross-validation** for all models  
✅ Conducted **hyperparameter tuning** for best performing model  
✅ Analyzed **feature importance** for tree-based models  
✅ Saved all models and comparison results  
✅ **Best model**: {best_model_name} with {results_df.iloc[0]['Test Accuracy']:.4f} test accuracy